In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load sparse codes
path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/train_sparse_codes.npz"

data = np.load(path)

Gamma = data["Gamma"]
labels = data["labels"]

print("Gamma shape:", Gamma.shape)
print("Labels shape:", labels.shape)

classes = np.unique(labels)

plt.figure(figsize=(14,6))

for cls in classes:

    # Select patches belonging to this class
    Gamma_cls = Gamma[:, labels == cls]

    # Mean absolute value for each sparse coefficient
    mean_abs_coefficients = np.mean(
        np.abs(Gamma_cls),
        axis=1
    )

    # x = coefficient index
    # y = mean absolute coefficient value
    plt.plot(
        np.arange(Gamma.shape[0]),
        mean_abs_coefficients,
        label=f"Class {cls}"
    )

plt.xlabel("Sparse Coefficient Index")
plt.ylabel("Mean |Sparse Coefficient|")
plt.title("Mean Absolute Sparse Coefficient Value per Index")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

classes = np.unique(labels)

class_avg_coeff = {}

for cls in classes:
    Gamma_cls = Gamma[:, labels == cls]

    # Average activation of each coefficient
    mean_coeff = np.mean(np.abs(Gamma_cls), axis=1)

    class_avg_coeff[cls] = mean_coeff

    print(
        f"Class {cls}:",
        "mean =", mean_coeff.mean(),
        "max =", mean_coeff.max(),
        "non-zero atoms =", np.sum(mean_coeff > 1e-6)
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load sparse codes
path = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/train_sparse_codes.npz"

data = np.load(path)

Gamma = data["Gamma"]
labels = data["labels"]

print("Gamma shape:", Gamma.shape)
print("Labels shape:", labels.shape)

classes = np.unique(labels)

# Store results
class_mean_coefficients = {}

for cls in classes:

    # Select patches belonging to this class
    Gamma_cls = Gamma[:, labels == cls]

    # Mean absolute coefficient value for each coefficient index
    mean_abs_coefficients = np.mean(
        np.abs(Gamma_cls),
        axis=1
    )

    class_mean_coefficients[cls] = mean_abs_coefficients


    # Get top 10 coefficients
    top10_idx = np.argsort(mean_abs_coefficients)[::-1][:10]


    print("\n================================")
    print(f"Class {cls}")
    print("Number of patches:", Gamma_cls.shape[1])
    print("--------------------------------")
    print("Top 10 sparse coefficient indices:")

    for idx in top10_idx:
        print(
            f"Index {idx:5d}  |  Mean |Coefficient| = {mean_abs_coefficients[idx]:.6f}"
        )


    # -----------------------------
    # Plot for this class
    # -----------------------------
    plt.figure(figsize=(14,5))

    plt.plot(
        np.arange(Gamma.shape[0]),
        mean_abs_coefficients
    )

    plt.xlabel("Sparse Coefficient Index")
    plt.ylabel("Mean |Sparse Coefficient|")
    plt.title(
        f"Class {cls}: Mean Absolute Sparse Coefficient Value"
    )

    plt.grid(True)
    plt.tight_layout()

    plt.show()

In [ ]:
import numpy as np

print("Average signed coefficient values for top 10 coefficients per class")

for cls in classes:

    # Select patches belonging to this class
    Gamma_cls = Gamma[:, labels == cls]

    # Get mean absolute coefficients (same as previous cell)
    mean_abs_coefficients = np.mean(
        np.abs(Gamma_cls),
        axis=1
    )

    # Get top 10 coefficient indices from previous criterion
    top10_idx = np.argsort(mean_abs_coefficients)[::-1][:10]

    print("\n================================")
    print(f"Class {cls}")
    print("--------------------------------")
    print("Coefficient Index | Mean Signed Coefficient")

    for idx in top10_idx:

        # Average without absolute value
        mean_signed = np.mean(
            Gamma_cls[idx, :]
        )

        print(
            f"{idx:5d}              | {mean_signed:.6f}"
        )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load(
    "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/train_sparse_codes.npz",
    allow_pickle=True
)

Gamma = data["Gamma"]
labels = data["labels"]

# Normal class
normal_idx = labels == 2
Gamma_normal = Gamma[:, normal_idx]

# Atom usage frequency
atom_usage = np.count_nonzero(Gamma_normal, axis=1)

top_atoms = np.argsort(atom_usage)[::-1]

print("Top atoms:")
for i in top_atoms[:20]:
    print(
        f"Atom {i}: "
        f"{atom_usage[i]} patches "
        f"({100*atom_usage[i]/Gamma_normal.shape[1]:.2f}%)"
    )

In [ ]:
top_k = 10

fig, axes = plt.subplots(
    2,
    5,
    figsize=(18,7)
)

axes = axes.flatten()

for ax, atom in zip(
    axes,
    top_atoms[:top_k]
):

    # coefficients of this atom across normal patches
    coefficients = Gamma_normal[atom, :]

    # only active coefficients
    active_coeffs = coefficients[
        coefficients != 0
    ]

    ax.hist(
        active_coeffs,
        bins=50
    )

    ax.set_title(
        f"Atom {atom}\n"
        f"Used: {len(active_coeffs)} patches"
    )

    ax.set_xlabel("Coefficient value")
    ax.set_ylabel("Count")


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gamma_indices = np.arange(0, 117386)

# Extract sparse codes
Gamma_patches = Gamma[:, gamma_indices]

print("Sparse codes shape:", Gamma_patches.shape)

# Absolute coefficients
Gamma_abs = np.abs(Gamma_patches)

# Sum contribution of each atom across all patches
atom_contribution = np.sum(
    Gamma_abs,
    axis=1
)

print("Atom contribution shape:", atom_contribution.shape)


# Only plot atoms that contributed
active_atoms = np.where(atom_contribution > 0)[0]
active_values = atom_contribution[active_atoms]


plt.figure(figsize=(18,5))

plt.stem(
    active_atoms,
    active_values
)

plt.xlabel("Dictionary atom index")
plt.ylabel("Sum of |sparse coefficient|")
plt.title(
    f"Aggregated sparse representation\n"
    f"{len(gamma_indices)} patches, "
    f"{len(active_atoms)} active atoms"
)

plt.grid(True)
plt.show()

In [ ]:
# Get top 10 contributing atoms
top_10_atoms = np.argsort(atom_contribution)[::-1][:10]

print("Top 10 atoms by summed |coefficient|:\n")

for atom in top_10_atoms:
    print(
        f"Atom {atom}: "
        f"Total |coefficient| = {atom_contribution[atom]:.6f}"
    )